# CRYSTALS-Kyber με Πλήρη Μαθηματική Ανάλυση

**Μετα-Κβαντικός Μηχανισμός Ενθυλάκωσης Κλειδιού | ML-KEM / FIPS 203**

Αυτό το notebook διδάσκει τα **πλήρη μαθηματικά** του αλγορίθμου CRYSTALS-Kyber.
Κάθε έννοια εξηγείται πρώτα και στη συνέχεια υλοποιείται αμέσως σε Python.

**Παράμετροι παιχνιδιού** που ακολουθούν την ίδια δομή με το FIPS 203, απλώς με μικρότερους αριθμούς:

| Σύμβολο | Τιμή | Σημασία |
|--------|-------|---------|
| q | 17 | Πρώτος αριθμός mod, όλοι οι συντελεστές ανήκουν στο {0 … 16} |
| n | 8 | Βαθμός πολυωνύμου, modulus δακτυλίου είναι X⁸ + 1 |
| k | 2 | Διάσταση module, ο A είναι 2×2, τα διανύσματα έχουν 2 πολυώνυμα |
| η | 2 | Όριο θορύβου CBD, μυστικοί συντελεστές στο {−2,−1,0,1,2} |

**Πώς να χρησιμοποιήσετε αυτό το notebook:**
1. Εκτελέστε πρώτα το **Module 0: Καθολική Εγκατάσταση**. Ορίζει όλες τις κοινές σταθερές.
2. Ακολουθήστε τα modules με τη σειρά. Κάθε κελί βασίζεται στο προηγούμενο.
3. Όλες οι πράξεις γίνονται mod **q = 17** εκτός αν αναφέρεται διαφορετικά.

> **Όλοι οι αριθμοί είναι αυτο-συνεπείς και επαληθεύονται από τους ελέγχους (assertions) σε κάθε κελί.** Ο δακτύλιος, ο αρνητικοκυκλικός NTT (μέσω στρέψης psi) και η ροή KeyGen->Ενθυλάκωση->Αποενθυλάκωση αντικατοπτρίζουν τη δομή του FIPS 203 σε κλίμακα παιχνιδιού.

## Module 0: Καθολική Εγκατάσταση

> **Εκτελέστε αυτό το κελί πρώτα.** Όλα τα άλλα κελιά εξαρτώνται από αυτές τις σταθερές.
> Μην τροποποιήσετε τις τιμές παρακάτω. Είναι η επαληθευμένη βάση αναφοράς για όλες τις ασκήσεις.

Οι σταθερές χωρίζονται σε τρεις ομάδες:
- **Παράμετροι** που αποτελούν τις μαθηματικές ρυθμίσεις αυτής της έκδοσης Kyber
- **Τιμές Δημιουργίας Κλειδιού:** μυστικό κλειδί s, σφάλμα e, ρίζες NTT, δημόσιο κλειδί t
- **Τιμές Ενθυλάκωσης και Αποενθυλάκωσης:** κρυπτοκείμενο u, v και ανάκτηση w

In [ ]:
# Setup: toy parameters, NTT twist tables, and reference values (run this cell first)

# Toy parameters:
Q     = 17   # prime modulus
N     = 8    # polynomial degree, ring is X^8 + 1 (negacyclic)
K     = 2    # module dimension
ETA   = 2    # CBD noise parameter eta
N_INV = 15   # n^-1: 8 x 15 = 120 = 1 mod 17

# NTT roots (omega = 9 is a primitive n-th = 8th root of unity)
ROOTS     = [1, 9, 13, 15, 16, 8, 4, 2]
INV_ROOTS = [1, 2,  4,  8, 16, 15, 13, 9]

# Negacyclic twist tables (psi = 3 is a primitive 2n-th = 16th root, psi^8 = -1).
# These turn the plain (cyclic) NTT into multiplication in Z_17[X]/(X^8 + 1).
PSI         = 3
PSI_INV     = 6                                  # 3 x 6 = 18 = 1 mod 17
PSI_POW     = [1, 3, 9, 10, 13, 5, 15, 11]      # psi^i
PSI_INV_POW = [1, 6, 2, 12, 4, 7, 8, 14]  # psi^-i

# Matrix A (NTT domain, generated from public seed rho)
A = [
    [[2,11,8,1,15,3,3,6],  [6,6,9,9,12,12,15,15]],
    [[7,14,3,10,5,12,1,8], [3,7,2,9,14,4,11,6  ]]
]

# Secret key s and error e (CBD output, normal domain)
S0 = [1, 0, 0, 1, 0, 0, 0, 1]
S1 = [0, 1, 0, 0, 1, 0, 0, 0]
E0 = [1, 0, 0, 0, 0, 0, 0, 0]
E1 = [0, 0, 1, 0, 0, 0, 0, 0]

# NTT domain forms of s and e (negacyclic NTT)
S0_NTT = [5, 3, 0, 9, 14, 16, 2, 10]
S1_NTT = [16, 14, 1, 15, 10, 11, 8, 10]
E0_NTT = [1, 1, 1, 1, 1, 1, 1, 1]
E1_NTT = [9, 15, 8, 2, 9, 15, 8, 2]

# Public key t (NTT domain)
T0 = [5, 16, 10, 9, 8, 11, 8, 7]
T1 = [7, 2, 10, 6, 15, 13, 13, 6]

# Encapsulation: fresh noise and message
R0    = [1, 0, 1, 0, 0, 1, 0, 0]
R1    = [0, 1, 0, 0, 1, 0, 1, 0]
E1_0  = [0, 0, 0, 1, 0, 0, 0, 0]
E1_1  = [0, 0, 0, 0, 0, 1, 0, 0]
E2    = [0, 0, 0, 0, 1, 0, 0, 0]
R0_NTT = [15, 5, 6, 13, 5, 10, 12, 10]
R1_NTT = [14, 6, 3, 6, 8, 3, 10, 1]
MSG   = [1, 0, 1, 0, 1, 0, 1, 0]
M_HAT = [8, 0, 8, 0, 8, 0, 8, 0]   # message encoded: bit 1 -> 8, bit 0 -> 0

# Ciphertext
U0 = [10, 8, 14, 4, 0, 4, 3, 7]
U1 = [10, 11, 10, 16, 2, 11, 4, 7]
V  = [14, 8, 3, 0, 5, 12, 11, 3]

# Decapsulation intermediate values
U0_NTT = [2, 0, 12, 10, 3, 1, 6, 12]
U1_NTT = [1, 10, 6, 11, 14, 7, 4, 10]
STU_N  = [6, 7, 10, 0, 13, 11, 4, 3]
W      = [8, 1, 10, 0, 9, 1, 7, 0]
DECODED= [1, 0, 1, 0, 1, 0, 1, 0]

print(f"Setup complete.  Q={Q}  N={N}  K={K}  ETA={ETA}  N_INV={N_INV}")
print(f"omega roots:  {ROOTS}")
print(f"psi^i twist:  {PSI_POW}")


## Module 1: Οι Μαθηματικές Βάσεις

### 1.1 Ο Δακτύλιος R_q = Z₁₇[X] / (X⁸ + 1)

Όλη η αριθμητική του Kyber γίνεται μέσα σε αυτόν τον δακτύλιο:
- **Κάθε πολυώνυμο έχει το πολύ 8 όρους**, δηλαδή συντελεστές από X⁰ έως X⁷
- **Όλοι οι συντελεστές είναι ακέραιοι mod q = 17** και ανήκουν στο {0, 1, …, 16}
- **Αρνητική κυκλική αναγωγή:** όταν ο πολλαπλασιασμός παράγει X^k με k ≥ 8, χρησιμοποιούμε την ταυτότητα
 `X⁸ ≡ −1 mod (X⁸+1)`, που δίνει:
 `X⁸ → −1`, `X⁹ → −X`, `X¹⁰ → −X²`, …, `X¹⁵ → −X⁷`

**Παράδειγμα:** ένας συντελεστής στο X¹⁰ μετασχηματίζεται ως `−1 × (συντελεστής του X²) mod 17`

### 1.2 Εύρεση της Πρωτόγονης Ρίζας g

Χρειαζόμαστε μια γεννήτρια g του Z*₁₇, δηλαδή ένα στοιχείο του οποίου οι δυνάμεις παράγουν και τα 16 μη μηδενικά υπόλοιπα mod 17.

**Έλεγχος:** το g πρέπει να έχει **τάξη 16**, δηλαδή g^16 ≡ 1 mod 17 ΚΑΙ g^k ≠ 1 για κάθε 1 ≤ k < 16.
Ισοδύναμα (αφού 16 = 2⁴, μοναδικός πρώτος παράγοντας είναι το 2): ελέγχουμε ότι g^8 ≠ 1 mod 17.

- g = 2 **αποτυγχάνει**: 2⁸ mod 17 = 1 → τάξη μόνο 8, όχι 16
- g = 3 **λειτουργεί**: 3⁸ mod 17 = 16 = −1 ≠ 1 → τάξη 16 ✓

### 1.3 Υπολογισμός των ψ, ω και των πινάκων ριζών

Από το g = 3 παράγουμε όλα όσα χρειαζόμαστε για το NTT:

```
ψ = g^((q−1)/(2n)) mod q = 3^((17−1)/(2×8)) = 3¹ mod 17 = 3
 Check: ψ^16 = 1 mod 17 ✓ (primitive 2n-th root of unity)
 Check: ψ^8 = 16 = −1 mod 17 ✓ (must not equal 1)

ω = ψ² mod 17 = 9
 Check: ω^8 = 1 mod 17 ✓ (primitive n-th root of unity)

ω⁻¹ = 2 επειδή 9 × 2 = 18 ≡ 1 mod 17
n⁻¹ = 15 επειδή 8 × 15 = 120 ≡ 1 mod 17

roots[k] = ω^k mod 17 = [1, 9, 13, 15, 16, 8, 4, 2]
inv_roots[k] = (ω⁻¹)^k mod 17 = [1, 2, 4, 8, 16, 15, 13, 9]
```

**Επαλήθευση:** `roots[k] × inv_roots[k] ≡ 1 mod 17` για κάθε k

### Άσκηση 1: Υλοποιήστε τη `modpow` και επαληθεύστε g, ψ, ω

Υλοποιήστε τη γρήγορη αριθμητική modular ύψωση σε δύναμη με τη μέθοδο **τετραγωνισμού-και-πολλαπλασιασμού**:

```
result = 1
while exp > 0:
 if exp is odd: result = result × base mod mod
 base = base² mod mod
 exp = exp // 2
```

Στη συνέχεια χρησιμοποιήστε την για να επαληθεύσετε την αλυσίδα g → ψ → ω → πίνακες ριζών.

In [ ]:
def modpow(base: int, exp: int, mod: int) -> int:
    """Γρήγορη modular ύψωση σε δύναμη (τετραγωνισμός-και-πολλαπλασιασμός)."""
    result, b = 1, base % mod
    while exp > 0:
        if exp & 1:            # αν exp περιττό
            result = result * b % mod
        b = b * b % mod        # τετραγωνισμός βάσης
        exp >>= 1              # exp = exp // 2
    return result

# Έλεγχος: g = 2 αποτυγχάνει
print("Λειτουργεί το g = 2;")
print(f"2^8  mod 17 = {modpow(2,  8, Q)}   (χρειάζεται ≠ 1 για πλήρη τάξη 16)")
print(f"2^16 mod 17 = {modpow(2, 16, Q)}   (τάξη είναι 8, όχι 16 → ΑΠΟΡΡΙΠΤΕΤΑΙ ✗)")

# Έλεγχος: g = 3 λειτουργεί
print("\nΛειτουργεί το g = 3;")
print("k    3^k mod 17")
elements = set()
for k in range(1, 17):
    v = modpow(3, k, Q)
    elements.add(v)
    mark = "  ← ταυτότητα, τάξη = 16 επιβεβαιώθηκε ✓" if v == 1 else ""
    print(f"  {k:2d}     {v:2d}{mark}")
print(f"\nΠαράχθηκαν {len(elements)} διαφορετικές τιμές → g = 3 ΑΠΟΔΕΚΤΟ ✓")

# Υπολογισμός ψ, ω, αντιστρόφων
psi       = modpow(3, (Q-1)//(2*N), Q)   # ψ = 3^1 mod 17 = 3
omega     = modpow(psi, 2, Q)             # ω = ψ² = 9
omega_inv = modpow(omega, Q-2, Q)         # ω⁻¹ via Fermat: ω^(q-2)
n_inv_v   = modpow(N, Q-2, Q)            # n⁻¹ = 8^15 mod 17 = 15

print(f"\nψ  = g^((q-1)/(2n)) = {psi}")
print(f"   ψ^16 = {modpow(psi, 16, Q)}  (πρέπει να είναι 1) ✓")
print(f"   ψ^8  = {modpow(psi,  8, Q)}  (πρέπει να είναι 16 = −1) ✓")
print(f"ω  = ψ² mod 17 = {omega}")
print(f"   ω^8  = {modpow(omega, 8, Q)}  (πρέπει να είναι 1) ✓")
print(f"ω⁻¹ = {omega_inv}  check: {omega} × {omega_inv} mod 17 = {omega*omega_inv%Q}")
print(f"n⁻¹ = {n_inv_v}  check: {N} × {n_inv_v} mod 17 = {N*n_inv_v%Q}")

# Κατασκευή και επαλήθευση πινάκων ριζών
fwd = [modpow(omega,     k, Q) for k in range(N)]
inv = [modpow(omega_inv, k, Q) for k in range(N)]
print(f"\nΕυθείες ρίζες:  {fwd}")
print(f"Αντίστροφες ρίζες:  {inv}")
check = [fwd[k] * inv[k] % Q for k in range(N)]
print(f"Έλεγχος γινομένου:  {check}  (όλα πρέπει να είναι 1)")

assert fwd == ROOTS,     "Αναντιστοιχία ευθειών ριζών!"
assert inv == INV_ROOTS, "Αναντιστοιχία αντίστροφων ριζών!"
print("\n✓ Και οι δύο πίνακες ριζών συμφωνούν με τις καθολικές σταθερές.")

## Module 2 Δειγματοληψία CBD: Δημιουργία s και e

### 2.1 Γιατί μικροί συντελεστές;

Η ασφάλεια του Kyber εξαρτάται από το μυστικό κλειδί **s** και το σφάλμα **e** να έχουν **μικρούς** συντελεστές.
Ο όρος θορύβου στην αποενθυλάκωση πρέπει να ικανοποιεί `|θόρυβος| < q/4 = 4.25` για σωστή αποκρυπτογράφηση.
Το CBD με η=2 εγγυάται συντελεστές στο {−2, −1, 0, +1, +2}, που είναι αρκετά μικροί.

### 2.2 Ο αλγόριθμος CBD (η = 2)

Είσοδος: ροή bytes από `SHAKE256(σ ‖ counter)`, διαβάζεται **LSB-πρώτο** (bit 0 του byte πρώτο).

Για κάθε 4 bits `(b₀, b₁, b₂, b₃)`:
```
coefficient = (b₀ + b₁) − (b₂ + b₃)
```
Αυτό ακολουθεί την **κεντρική διωνυμική κατανομή** με παράμετρο η=2:
τιμές −2, −1, 0, +1, +2 με πιθανότητες 1/16, 4/16, 6/16, 4/16, 1/16.

**Αρνητικοί συντελεστές** αποθηκεύονται ως θετικός αντιπρόσωπος mod q:
`−1 → 16`, `−2 → 15`

**Ένα byte → 2 συντελεστές. Τέσσερα bytes → ένα πολυώνυμο (n=8 συντελεστές).**

### 2.3 Πλήρης ιχνηλάτηση από το έγγραφο αναφοράς

```
s[0] input bytes: [0x01, 0x10, 0x00, 0x10]

Byte 0x01 = 00000001 (LSB-first: 1,0,0,0,0,0,0,0)
 coeff[0]: (b0+b1)−(b2+b3) = (1+0)−(0+0) = +1 → stored: 1
 coeff[1]: (b4+b5)−(b6+b7) = (0+0)−(0+0) = 0 → stored: 0

Byte 0x10 = 00010000 (LSB-first: 0,0,0,0,1,0,0,0)
 coeff[2]: (0+0)−(0+0) = 0 → stored: 0
 coeff[3]: (1+0)−(0+0) = +1 → stored: 1

Byte 0x00 → coeff[4]=0, coeff[5]=0
Byte 0x10 → coeff[6]=0, coeff[7]=+1

s[0] = [1, 0, 0, 1, 0, 0, 0, 1] ✓
```

### Άσκηση 2: Υλοποιήστε τη `cbd_sample`

Μετατρέψτε μια λίστα bytes σε συντελεστές πολυωνύμου χρησιμοποιώντας τον αλγόριθμο CBD.

In [ ]:
def cbd_sample(byte_list: list, q: int) -> list:
    """
    CBD η=2: μετατροπή bytes σε συντελεστές πολυωνύμου.
    Κάθε byte → 2 συντελεστές (4 bits έκαστος, LSB-πρώτο).
    Επιστρέφει len(byte_list) × 2 συντελεστές, καθένας στο {0 … q-1}.
    """
    coefficients = []
    for byte_val in byte_list:
        # Εξαγωγή όλων των 8 bits, LSB πρώτο
        bits = [(byte_val >> i) & 1 for i in range(8)]
        # Δύο ομάδες 4 bits ανά byte
        for pair in range(2):
            b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
            coeff = (b0 + b1) - (b2 + b3)   # κεντρική διωνυμική
            coefficients.append(coeff % q)  # αποθήκευση θετικού αντιπροσώπου
    return coefficients

print("CBD για bytes s[0] [0x01, 0x10, 0x00, 0x10]\n")
s0_bytes = [0x01, 0x10, 0x00, 0x10]
for idx, byte_val in enumerate(s0_bytes):
    bits = [(byte_val >> i) & 1 for i in range(8)]
    print(f"  Byte 0x{byte_val:02X} = {byte_val:08b}  (LSB-πρώτο: {bits})")
    for pair in range(2):
        b0, b1, b2, b3 = bits[pair*4 : pair*4+4]
        raw    = (b0+b1) - (b2+b3)
        stored = raw % Q
        print(f"    coeff[{idx*2+pair}]: ({b0}+{b1})−({b2}+{b3}) = {raw:+d}  →  αποθηκεύτηκε: {stored}")
    print()

s0 = cbd_sample([0x01, 0x10, 0x00, 0x10], Q)
s1 = cbd_sample([0x10, 0x00, 0x01, 0x00], Q)
e0 = cbd_sample([0x01, 0x00, 0x00, 0x00], Q)
e1 = cbd_sample([0x00, 0x01, 0x00, 0x00], Q)

print(f"s[0] = {s0}  αναμενόμενο: {S0}")
print(f"s[1] = {s1}  αναμενόμενο: {S1}")
print(f"e[0] = {e0}  αναμενόμενο: {E0}")
print(f"e[1] = {e1}  αναμενόμενο: {E1}")

assert s0==S0 and s1==S1 and e0==E0 and e1==E1
print("\n✓ Όλες οι έξοδοι CBD συμφωνούν με τις καθολικές σταθερές.")

# Ακραίες περιπτώσεις
print("\nΑκραίες περιπτώσεις:")
print(f"  0xFF → {cbd_sample([0xFF], Q)}  (1111|1111 → (1+1)−(1+1)=0 δύο φορές)")
print(f"  0x0F → {cbd_sample([0x0F], Q)}  (1111|0000 → +2 μετά 0)")
print(f"  0xF0 → {cbd_sample([0xF0], Q)}  (0000|1111 → 0 μετά −2=15)")

## Module 3: Ευθύς NTT (Cooley-Tukey) και η Αρνητικοκυκλική Στρέψη

### 3.1 Γιατί NTT;

Ο απλός πολλαπλασιασμός πολυωνύμων στο R_q απαιτεί **O(n2)** πράξεις.
Ο **Θεωρητικός Μετασχηματισμός Αριθμών** τα μετατρέπει σε χώρο όπου ο
πολλαπλασιασμός είναι **σημειακός**, μειώνοντας το κόστος σε **O(n log n)**.

### 3.2 Κυκλικό vs αρνητικοκυκλικό: γιατί χρειαζόμαστε το psi

Ένας απλός NTT με `omega` (πρωταρχική n-οστή ρίζα, εδώ omega = 9) υπολογίζει την
**κυκλική** συνέλιξη, δηλαδή πολλαπλασιασμό mod `X^n - 1`.
Όμως ο δακτύλιος του Kyber είναι **αρνητικοκυκλικός**: `Z_q[X]/(X^n + 1)`, όπου `X^n = -1`.

Για να γεφυρώσουμε τη διαφορά χρησιμοποιούμε το `psi`, μια πρωταρχική **2n-οστή** ρίζα με `psi^n = -1` (εδώ psi = 3):

```
ευθύς:      a_hat = NTT_omega( psi^i  * a[i] )     # προ-στρέψη, μετά κυκλικός NTT
αντίστροφος: a[i] = psi^-i * INTT_omega( a_hat )   # κυκλικός INTT, μετά μετα-στρέψη
```

Η προ-στρέψη με `psi^i` και η μετα-στρέψη με `psi^-i` είναι ακριβώς αυτό που μετατρέπει
τον κυκλικό μετασχηματισμό σε πολλαπλασιασμό στο `X^n + 1`. Με psi = 3: `psi^i = [1,3,9,10,13,5,15,11]`.

### 3.3 Πώς διαφέρει ο πραγματικός Kyber (σημείωση πιστότητας)

Ο πραγματικός Kyber (q = 3329, n = 256) **δεν έχει πρωταρχική 512η ρίζα**, οπότε ο NTT του
είναι **ατελής**: 7 στρώματα που σταματούν σε 128 ζεύγη βαθμού 1, ολοκληρωμένα από ένα βήμα `basemul`.
Το δικό μας παιχνίδι (q = 17, n = 8) **έχει** πρωταρχική 16η ρίζα (psi = 3), οπότε τρέχουμε
**πλήρη** NTT μέχρι μεμονωμένα σημεία. Ίδια ιδέα, διαφορετικό βάθος: μην υποθέσετε ότι ο NTT
του πραγματικού Kyber φτάνει σε μεμονωμένα σημεία — δεν φτάνει.

### 3.4 Μετάθεση αναστροφής bits

Πριν τα στρώματα πεταλούδας, ο πίνακας μετατίθεται αναστρέφοντας τον δυαδικό δείκτη.
Για n = 8: τα ζεύγη (1,4) και (3,6) εναλλάσσονται· τα (0, 2, 5, 7) παραμένουν.

### 3.5 Πράξη πεταλούδας

```
t = v * w mod q ; new_u = (u + t) mod q ; new_v = (u - t) mod q
```
με τρία στρώματα (len = 2, 4, 8), `step = n // len`, στροφέας `w = roots[j * step]`.

### 3.6 Πλήρης αρνητικοκυκλική ιχνηλάτηση για s[0] = [1,0,0,1,0,0,0,1]

```
Pre-twist by psi^i (psi=3, psi^i = [1, 3, 9, 10, 13, 5, 15, 11]):
  s[0]        = [1, 0, 0, 1, 0, 0, 0, 1]
  s[0]*psi^i  = [1, 0, 0, 10, 0, 0, 0, 11]
After bit-reversal: [1, 0, 0, 0, 0, 0, 10, 11]
Layer 1 (len=2, step=4):
  [0,1] u= 1 v= 0 w= 1 -> t= 0  new[0]= 1 new[1]= 1
  [2,3] u= 0 v= 0 w= 1 -> t= 0  new[2]= 0 new[3]= 0
  [4,5] u= 0 v= 0 w= 1 -> t= 0  new[4]= 0 new[5]= 0
  [6,7] u=10 v=11 w= 1 -> t=11  new[6]= 4 new[7]=16
  After layer 1: [1, 1, 0, 0, 0, 0, 4, 16]
Layer 2 (len=4, step=2):
  [0,2] u= 1 v= 0 w= 1 -> t= 0  new[0]= 1 new[2]= 1
  [1,3] u= 1 v= 0 w=13 -> t= 0  new[1]= 1 new[3]= 1
  [4,6] u= 0 v= 4 w= 1 -> t= 4  new[4]= 4 new[6]=13
  [5,7] u= 0 v=16 w=13 -> t= 4  new[5]= 4 new[7]=13
  After layer 2: [1, 1, 1, 1, 4, 4, 13, 13]
Layer 3 (len=8, step=1):
  [0,4] u= 1 v= 4 w= 1 -> t= 4  new[0]= 5 new[4]=14
  [1,5] u= 1 v= 4 w= 9 -> t= 2  new[1]= 3 new[5]=16
  [2,6] u= 1 v=13 w=13 -> t=16  new[2]= 0 new[6]= 2
  [3,7] u= 1 v=13 w=15 -> t= 8  new[3]= 9 new[7]=10
  After layer 3: [5, 3, 0, 9, 14, 16, 2, 10]
NTT(s[0]) = [5, 3, 0, 9, 14, 16, 2, 10]
```

### Άσκηση 3: Υλοποιήστε το `ntt` (προ-στρέψη + κυκλικός Cooley-Tukey)


In [ ]:
import math

def bit_reverse_permute(a: list) -> list:
    """Reorder elements by bit-reversed indices."""
    n    = len(a)
    bits = int(math.log2(n))
    r    = list(a)
    for i in range(n):
        rev = int(f'{i:0{bits}b}'[::-1], 2)
        if rev > i:
            r[i], r[rev] = r[rev], r[i]
    return r

def _cyclic(a: list, roots: list, q: int) -> list:
    """Plain Cooley-Tukey NTT. On its own this realizes CYCLIC convolution
    (mod X^n - 1). The psi-twist in ntt()/intt() upgrades it to the
    negacyclic ring mod X^n + 1 that Kyber actually uses."""
    n = len(a)
    a = bit_reverse_permute(a)
    length = 2
    while length <= n:
        step = n // length
        for bs in range(0, n, length):
            for j in range(length // 2):
                f, s = bs + j, bs + j + length // 2
                w    = roots[j * step]
                t    = a[s] * w % q
                a[f], a[s] = (a[f] + t) % q, (a[f] - t) % q
        length <<= 1
    return a

def ntt(poly: list, roots: list, q: int) -> list:
    """Negacyclic forward NTT: pre-twist by psi^i, then the cyclic transform."""
    n = len(poly)
    twisted = [poly[i] * PSI_POW[i] % q for i in range(n)]
    return _cyclic(twisted, roots, q)

# Confirm against the global constants
print("s[0]              =", S0)
print("s[0] * psi^i      =", [S0[i]*PSI_POW[i] % Q for i in range(N)])
print("NTT(s[0])         =", ntt(S0, ROOTS, Q), " expected", S0_NTT)
assert ntt(S0, ROOTS, Q) == S0_NTT
assert ntt(S1, ROOTS, Q) == S1_NTT
assert ntt(E0, ROOTS, Q) == E0_NTT
assert ntt(E1, ROOTS, Q) == E1_NTT
print("\nAll NTT results match the global constants.")


## Module 4: Αντίστροφος NTT

### 4.1 Δομή

Ο INTT επαναχρησιμοποιεί την **ίδια κυκλική πεταλούδα** με τον ευθύ, με τρία βήματα:

1. Εκτέλεση του κυκλικού μετασχηματισμού με `inv_roots` αντί για `roots`.
2. Πολλαπλασιασμός κάθε συντελεστή με `n^-1 = 15`.
3. **Μετα-στρέψη** με `psi^-i` (psi^-1 = 6) για αναίρεση της ευθείας στρέψης `psi^i`.

Τα βήματα 2-3 κάνουν αυτόν τον αντίστροφο πραγματικά αρνητικοκυκλικό, ώστε
`intt(ntt(a)) == a` στο `Z_17[X]/(X^8 + 1)`.

### 4.2 Ιχνηλάτηση κυκλικής επαλήθευσης για NTT(s[0])

```
Input  NTT(s[0]) = [5, 3, 0, 9, 14, 16, 2, 10]
After bit-reversal: [5, 14, 0, 2, 3, 16, 9, 10]
Layer 1 (len=2, step=4):
  [0,1] u= 5 v=14 w= 1 -> t=14  new[0]= 2 new[1]= 8
  [2,3] u= 0 v= 2 w= 1 -> t= 2  new[2]= 2 new[3]=15
  [4,5] u= 3 v=16 w= 1 -> t=16  new[4]= 2 new[5]= 4
  [6,7] u= 9 v=10 w= 1 -> t=10  new[6]= 2 new[7]=16
  After layer 1: [2, 8, 2, 15, 2, 4, 2, 16]
Layer 2 (len=4, step=2):
  [0,2] u= 2 v= 2 w= 1 -> t= 2  new[0]= 4 new[2]= 0
  [1,3] u= 8 v=15 w= 4 -> t= 9  new[1]= 0 new[3]=16
  [4,6] u= 2 v= 2 w= 1 -> t= 2  new[4]= 4 new[6]= 0
  [5,7] u= 4 v=16 w= 4 -> t=13  new[5]= 0 new[7]= 8
  After layer 2: [4, 0, 0, 16, 4, 0, 0, 8]
Layer 3 (len=8, step=1):
  [0,4] u= 4 v= 4 w= 1 -> t= 4  new[0]= 8 new[4]= 0
  [1,5] u= 0 v= 0 w= 2 -> t= 0  new[1]= 0 new[5]= 0
  [2,6] u= 0 v= 0 w= 4 -> t= 0  new[2]= 0 new[6]= 0
  [3,7] u=16 v= 8 w= 8 -> t=13  new[3]=12 new[7]= 3
  After layer 3: [8, 0, 0, 12, 0, 0, 0, 3]
Before scaling: [8, 0, 0, 12, 0, 0, 0, 3]
Multiply by n^-1 = 15: [1, 0, 0, 10, 0, 0, 0, 11]
Post-twist by psi^-i (psi^-1=6): [1, 0, 0, 1, 0, 0, 0, 1]
```

### Άσκηση 4: Υλοποιήστε το `intt`


In [ ]:
def intt(poly: list, inv_roots: list, q: int, n_inv: int) -> list:
    """Negacyclic inverse NTT:
        1. cyclic transform with inv_roots
        2. scale every coefficient by n^-1
        3. post-twist by psi^-i  (undoes the forward psi^i twist)."""
    n      = len(poly)
    a      = _cyclic(poly, inv_roots, q)
    scaled = [x * n_inv % q for x in a]
    return [scaled[i] * PSI_INV_POW[i] % q for i in range(n)]

print("INTT(NTT(s[0])) =", intt(S0_NTT, INV_ROOTS, Q, N_INV), " expected", S0)
assert intt(S0_NTT, INV_ROOTS, Q, N_INV) == S0
assert intt(S1_NTT, INV_ROOTS, Q, N_INV) == S1
assert intt(E0_NTT, INV_ROOTS, Q, N_INV) == E0
assert intt(E1_NTT, INV_ROOTS, Q, N_INV) == E1
print("All INTT round-trips match the global constants.")


## Module 5: Δημόσιο Κλειδί  t_hat = A_hat . s_hat + e_hat

### 5.1 Όλα παραμένουν στον χώρο NTT

Μόλις τα s, e μετασχηματιστούν (s_hat, e_hat) και το A είναι σε χώρο NTT (A_hat),
το δημόσιο κλειδί χρειάζεται μόνο **σημειακές** πράξεις, οπότε κανένα INTT εδώ:

```
t_hat[0] = A_hat[0][0] (x) s_hat[0] + A_hat[0][1] (x) s_hat[1] + e_hat[0]
t_hat[1] = A_hat[1][0] (x) s_hat[0] + A_hat[1][1] (x) s_hat[1] + e_hat[1]
```

`(x)` = σημειακός πολλαπλασιασμός mod q, `+` = σημειακή πρόσθεση mod q.

### 5.2 Αναλυτική στήλη για t_hat[0]

Με το αρνητικοκυκλικό s_hat[0] = [5, 3, 0, 9, 14, 16, 2, 10]:

Και οι 8 συντελεστές του t_hat[0]:

| i | A[0][0]xs[0] | A[0][1]xs[1] | +e[0] | t_hat[0][i] |
|---|--------------|--------------|-------|-------------|
| 0 | 2x5=10 | 6x16=11 | 1 | (10+11+1) mod 17 = **5** |
| 1 | 11x3=16 | 6x14=16 | 1 | (16+16+1) mod 17 = **16** |
| 2 | 8x0=0 | 9x1=9 | 1 | (0+9+1) mod 17 = **10** |
| 3 | 1x9=9 | 9x15=16 | 1 | (9+16+1) mod 17 = **9** |
| 4 | 15x14=6 | 12x10=1 | 1 | (6+1+1) mod 17 = **8** |
| 5 | 3x16=14 | 12x11=13 | 1 | (14+13+1) mod 17 = **11** |
| 6 | 3x2=6 | 15x8=1 | 1 | (6+1+1) mod 17 = **8** |
| 7 | 6x10=9 | 15x10=14 | 1 | (9+14+1) mod 17 = **7** |

`t_hat[0] = [5, 16, 10, 9, 8, 11, 8, 7]`   `t_hat[1] = [7, 2, 10, 6, 15, 13, 13, 6]`

### Άσκηση 5: Υλοποιήστε το `keygen_t`


In [ ]:
def pointwise_mul(a: list, b: list, q: int) -> list:
    """Σημειακός πολλαπλασιασμός δύο πολυωνύμων mod q."""
    return [x * y % q for x, y in zip(a, b)]

def pointwise_add(a: list, b: list, q: int) -> list:
    """Σημειακή πρόσθεση δύο πολυωνύμων mod q."""
    return [(x + y) % q for x, y in zip(a, b)]

def keygen_t(A_hat, s_ntt, e_ntt, q):
    """
    Υπολογισμός t̂ = Â·ŝ + ê στον χώρο NTT (σημειακές πράξεις).
    A_hat:  k×k πίνακας πολυωνύμων χώρου NTT
    s_ntt:  k-διάνυσμα πολυωνύμων χώρου NTT
    e_ntt:  k-διάνυσμα πολυωνύμων χώρου NTT
    Επιστρέφει: t̂ ως k-διάνυσμα πολυωνύμων χώρου NTT
    """
    k = len(A_hat)
    n = len(A_hat[0][0])
    t_hat = []
    for row in range(k):
        acc = [0] * n                              # συσσωρευτής για αυτή τη γραμμή
        for col in range(k):
            prod = pointwise_mul(A_hat[row][col], s_ntt[col], q)
            acc  = pointwise_add(acc, prod, q)
        t_hat.append(pointwise_add(acc, e_ntt[row], q))
    return t_hat

# Αναλυτική ιχνηλάτηση για t̂[0]
print("=== Κατασκευή t̂[0] βήμα προς βήμα ===\n")
p00 = pointwise_mul(A[0][0], S0_NTT, Q)
p01 = pointwise_mul(A[0][1], S1_NTT, Q)
print(f"Â[0][0] ⊙ ŝ[0] = {p00}")
print(f"Â[0][1] ⊙ ŝ[1] = {p01}")
s00 = pointwise_add(p00, p01, Q)
t0  = pointwise_add(s00, E0_NTT, Q)
print(f"άθροισμα         = {s00}")
print(f"+ ê[0]           = {t0}")
print(f"Αναμενόμενο t̂[0]:   {T0}")
assert t0 == T0

print("\n=== Κατασκευή t̂[1] ===\n")
t1 = pointwise_add(
    pointwise_add(pointwise_mul(A[1][0], S0_NTT, Q),
                  pointwise_mul(A[1][1], S1_NTT, Q), Q),
    E1_NTT, Q)
print(f"t̂[1] = {t1}  αναμενόμενο {T1}")
assert t1 == T1

t_hat = keygen_t(A, [S0_NTT, S1_NTT], [E0_NTT, E1_NTT], Q)
assert t_hat[0] == T0 and t_hat[1] == T1
print("\n✓ Το δημόσιο κλειδί t̂ συμφωνεί με τις καθολικές σταθερές.")
print(f"\npk = (t̂, ρ):  t̂[0] = {t_hat[0]}")
print(f"               t̂[1] = {t_hat[1]}")
print(f"sk = ŝ:         ŝ[0] = {S0_NTT}")
print(f"                ŝ[1] = {S1_NTT}")

## Module 6: Ενθυλάκωση

### 6.1 Επισκόπηση

Ο Βόβος κατέχει το δημόσιο κλειδί `pk = (t̂, ρ)`. Επιλέγει μήνυμα `m` και νέο θόρυβο `r, e1, e2`,
και παράγει κρυπτοκείμενο `(u, v)` που μόνο η Αλίκη (με μυστικό κλειδί ŝ) μπορεί να αποκρυπτογραφήσει.

```
u = INTT(Â^T · r̂) + e1
v = INTT(t̂^T · r̂) + e2 + m̂
```

**Ανάστροφος:** `Â^T[i][j] = Â[j][i]`, όπου οι γραμμές και οι στήλες του A εναλλάσσονται.

**Κωδικοποίηση μηνύματος:** κάθε bit του m κλιμακώνεται στο μέσο του Z_q:
```
bit 0 → 0 (near zero)
bit 1 → ⌊q/2⌋ = 8 (near the modulus midpoint)
```
Αυτό μεγιστοποιεί την απόσταση από λανθασμένη αποκωδικοποίηση.

### 6.2 Γιατί ο ανάστροφος;

Η αλγεβρική απαλοιφή στην αποενθυλάκωση απαιτεί:

```
v − ŝ^T · u = m̂ + small noise terms
```

Για να λειτουργήσει αυτό, ο Βόβος πρέπει να πολλαπλασιάζει με `Â^T` (όχι `Â`) κατά τον υπολογισμό του `u`.
Έτσι ο όρος `A·s` μέσα στο `t` απαλείφεται με τον όρο `A^T·r` μέσα στο `u`.

### 6.3 Υπολογισμός u, πλήρης ιχνηλάτηση για u[0]

```
Â^T row 0 = [Â[0][0], Â[1][0]] (column 0 of A, not row 0)

Â[0][0] ⊙ r̂[0]: [30→13, 55→4, 48→14, 13, 75→7, 30→13, 36→2, 60→9]
NTT-domain, reduced   = [13, 4, 14, 13, 7, 13, 2, 9]

Â[1][0] ⊙ r̂[1]: [98→13, 84→16, 9, 60→9, 40→6, 36→2, 10, 8]
NTT-domain, reduced   = [13, 16, 9, 9, 6, 2, 10, 8]

NTT sum: [9, 3, 6, 5, 13, 15, 12, 0]

INTT([9, 3, 6, 5, 13, 15, 12, 0]) = [10, 8, 14, 3, 0, 4, 3, 7]

+ e1[0] = [0, 0, 0, 1, 0, 0, 0, 0]: u[0] = [10, 8, 14, 4, 0, 4, 3, 7] ✓
```

### Άσκηση 6: Υλοποιήστε τη `encapsulate`

In [ ]:
def encode_message(msg: list, q: int) -> list:
    """Κωδικοποίηση κάθε bit: 0 → 0,  1 → ⌊q/2⌋."""
    return [bit * (q // 2) for bit in msg]

def matvec_ntt(M, v_ntt, q):
    """Πολλαπλασιασμός πίνακα-διανύσματος στον χώρο NTT (σημειακά ανά συντελεστή)."""
    k = len(M); n = len(M[0][0])
    result = []
    for row in range(k):
        acc = [0] * n
        for col in range(k):
            for i in range(n):
                acc[i] = (acc[i] + M[row][col][i] * v_ntt[col][i]) % q
        result.append(acc)
    return result

def encapsulate(A_hat, t_hat, r, e1, e2, msg, roots, inv_roots, q, k, n_inv):
    """
    Ενθυλάκωση Kyber.
    Επιστρέφει (u_list, v) όπου u_list είναι k-διάνυσμα πολυωνύμων κανονικού χώρου.
    """
    n = len(r[0])

    # NTT των πολυωνύμων r
    r_ntt = [ntt(rr, roots, q) for rr in r]

    # Ανάστροφος A: A_T[i][j] = A_hat[j][i]
    A_T = [[A_hat[j][i] for j in range(k)] for i in range(k)]

    # u = INTT(Â^T · r̂) + e1
    ATr_ntt = matvec_ntt(A_T, r_ntt, q)
    u = []
    for row in range(k):
        u_row = intt(ATr_ntt[row], inv_roots, q, n_inv)
        u.append([(u_row[i] + e1[row][i]) % q for i in range(n)])

    # v = INTT(t̂^T · r̂) + e2 + m̂
    tTr_ntt = [sum(t_hat[col][i] * r_ntt[col][i] for col in range(k)) % q
               for i in range(n)]
    v_pre = intt(tTr_ntt, inv_roots, q, n_inv)
    m_enc = encode_message(msg, q)
    v = [(v_pre[i] + e2[i] + m_enc[i]) % q for i in range(n)]

    return u, v

print("=== Ενθυλάκωση ===\n")
print(f"Μήνυμα m     = {MSG}")
print(f"Κωδικοποιημένο m̂    = {M_HAT}  (bit→8 αν 1, bit→0 αν 0)\n")

u_result, v_result = encapsulate(
    A, [T0, T1], [R0, R1], [E1_0, E1_1], E2, MSG,
    ROOTS, INV_ROOTS, Q, K, N_INV
)

print(f"u[0] = {u_result[0]}  αναμενόμενο {U0}")
print(f"u[1] = {u_result[1]}  αναμενόμενο {U1}")
print(f"v    = {v_result}  αναμενόμενο {V}")

assert u_result[0]==U0 and u_result[1]==U1 and v_result==V
print("\n✓ Το κρυπτοκείμενο (u, v) συμφωνεί με τις καθολικές σταθερές.")

## Module 7: Αποενθυλάκωση

### 7.1 Υπολογισμός της Αλίκης

Η Αλίκη κατέχει το μυστικό κλειδί `ŝ` και λαμβάνει κρυπτοκείμενο `(u, v)`:

```
w = v − INTT(ŝ^T · NTT(u))
```

Στη συνέχεια κάθε συντελεστής του w αποκωδικοποιείται σε bit με βάση την **κυκλική απόσταση**:
- `dist_to_0 = min(w[i], q − w[i])`
- `dist_to_q/2 = min(|w[i]−8|, q − |w[i]−8|)`
- **bit = 1** if `dist_to_q/2 < dist_to_0`, else **bit = 0**

### 7.2 Γιατί λειτουργεί η άλγεβρα: η απαλοιφή

Αναπτύσσοντας αλγεβρικά τα v και u:
```
v = INTT(t̂^T·r̂) + e2 + m̂
 = INTT((Â·ŝ + ê)^T·r̂) + e2 + m̂
 = INTT(ŝ^T·Â^T·r̂) + INTT(ê^T·r̂) + e2 + m̂

ŝ^T·u = INTT(ŝ^T·(Â^T·r̂ + NTT(e1)))
 = INTT(ŝ^T·Â^T·r̂) + INTT(ŝ^T·NTT(e1))

w = v − ŝ^T·u
 = m̂ + INTT(ê^T·r̂) + e2 − INTT(ŝ^T·NTT(e1))
 └──────────────────────────────────────────┘
 NOISE TERMS ONLY
```

Οι όροι `INTT(ŝ^T·Â^T·r̂)` απαλείφονται ακριβώς.
Παραμένει μόνο θόρυβος, και επειδή τα s, e, r, e1, e2 έχουν μικρούς συντελεστές, ο θόρυβος είναι μικρός.

### 7.3 Προϋπολογισμός θορύβου

```
i w[i] m̂[i] noise = w[i]−m̂[i] |noise| < q/4=4.25?
0 8 8 +0 0 < 4.25 ✓
1 1 0 +1 1 < 4.25 ✓
2 10 8 +2 2 < 4.25 ✓
3 0 0 +0 0 < 4.25 ✓
4 9 8 +1 1 < 4.25 ✓
5 1 0 +1 1 < 4.25 ✓
6 7 8 −1 (7−17=−1) 1 < 4.25 ✓
7 0 0 +0 0 < 4.25 ✓
Max noise = 2. All within budget. Decryption succeeds.
```

### Άσκηση 7: Υλοποιήστε τη `decapsulate`

In [ ]:
def decode_bit(coeff: int, q: int) -> int:
    """
    Αποκωδικοποίηση ενός συντελεστή σε bit με κυκλική απόσταση.
    Πιο κοντά στο 0     → bit 0
    Πιο κοντά στο q//2  → bit 1
    """
    d0  = min(coeff,          q - coeff)                   # απόσταση από 0
    dq2 = min(abs(coeff-q//2), q - abs(coeff-q//2))        # απόσταση από q/2
    return 1 if dq2 < d0 else 0

def decapsulate(s_hat, u, v, roots, inv_roots, q, k, n_inv):
    """
    Αποενθυλάκωση Kyber.
    Επιστρέφει τη λίστα αποκωδικοποιημένων bits.
    """
    n = len(u[0])

    # Μετασχηματισμός u στον χώρο NTT
    u_ntt = [ntt(uu, roots, q) for uu in u]

    # ŝ^T · û (εσωτερικό γινόμενο στον χώρο NTT)
    stu_ntt = [
        sum(s_hat[j][i] * u_ntt[j][i] for j in range(k)) % q
        for i in range(n)
    ]

    # INTT πίσω στον κανονικό χώρο
    stu_n = intt(stu_ntt, inv_roots, q, n_inv)

    # w = v − ŝ^T·u
    w = [(v[i] - stu_n[i]) % q for i in range(n)]

    # Αποκωδικοποίηση κάθε συντελεστή σε bit
    return [decode_bit(c, q) for c in w]

print("=== Αποενθυλάκωση ===\n")
u0_ntt = ntt(U0, ROOTS, Q)
u1_ntt = ntt(U1, ROOTS, Q)
print(f"NTT(u[0]) = {u0_ntt}  αναμενόμενο {U0_NTT}")
print(f"NTT(u[1]) = {u1_ntt}  αναμενόμενο {U1_NTT}")
assert u0_ntt==U0_NTT and u1_ntt==U1_NTT

stu_ntt = [(S0_NTT[i]*u0_ntt[i] + S1_NTT[i]*u1_ntt[i]) % Q for i in range(N)]
stu_n   = intt(stu_ntt, INV_ROOTS, Q, N_INV)
print(f"\nINTT(ŝ^T·û) = {stu_n}  αναμενόμενο {STU_N}")
assert stu_n == STU_N

w = [(V[i] - stu_n[i]) % Q for i in range(N)]
print(f"\nw = v − ŝ^T·u = {w}  αναμενόμενο {W}")
assert w == W

print("\n=== Προϋπολογισμός Θορύβου ===")
print(f"{'i':>3} {'w[i]':>6} {'m̂[i]':>6} {'θόρυβος':>10} {'|θόρ.|':>8} {'<q/4?':>8} {'bit':>5}")
decoded = []
for i in range(N):
    noise = (w[i] - M_HAT[i]) % Q
    if noise > Q // 2: noise -= Q
    ok  = "✓" if abs(noise) < Q/4 else "✗ ΑΠΟΤΥΧΙΑ"
    bit = decode_bit(w[i], Q)
    print(f"{i:>3} {w[i]:>6} {M_HAT[i]:>6} {noise:>+8} {abs(noise):>9} {Q/4:>7.2f} {ok} {bit:>5}")
    decoded.append(bit)

assert decoded == DECODED
print(f"\nΑποκωδικοποιημένο:  {decoded}")
print(f"Αρχικό: {MSG}")
print("\n✓ Αποενθυλάκωση επιτυχής: το μήνυμα ανακτήθηκε σωστά.")

## Module 8: Πλήρης Δοκιμή Συστήματος

### 8.1 Η πλήρης αλυσίδα KeyGen -> Ενθυλάκωση -> Αποενθυλάκωση

```
KeyGen:       s_hat, e_hat = NTT(s), NTT(e);  t_hat = A * s_hat + e_hat
Ενθυλάκωση:   r_hat = NTT(r);  u = INTT(A^T * r_hat) + e1;  v = INTT(t^T * r_hat) + e2 + encode(m)
Αποενθυλάκωση: w = v - INTT(s^T * NTT(u));  m' = decode(w);  assert m' == m
```

### 8.2 Γιατί ο Kyber είναι μετα-κβαντικά ασφαλής (με ακρίβεια)

Ο αλγόριθμος Shor σπάει RSA και ECDSA επειδή αυτά ανάγονται σε πρόβλημα **κρυφής
υποομάδας / εύρεσης περιόδου**: η `f(x) = a^x mod N` είναι περιοδική και ο Κβαντικός
Μετασχηματισμός Fourier εξάγει την περίοδο αποδοτικά.

Το Module-LWE **δεν είναι καθόλου πρόβλημα εύρεσης περιόδου**. Δεν υπάρχει κρυφή περίοδος
για να εντοπίσει ο QFT, οπότε η προσέγγιση του Shor δεν έχει πού να δράσει. Αυτό είναι
ιδιότητα της **δομής** του προβλήματος, όχι απλώς του θορύβου: είναι λάθος να πούμε "ο
θόρυβος καταστρέφει μια περίοδο που θα εκμεταλλευόταν ο QFT", γιατί δεν υπάρχει περίοδος
εξαρχής. Ο μικρός θόρυβος `e` κάνει την ανάκτηση του `s` δύσκολη (κλασικά και κβαντικά),
αλλά ο λόγος που αποτυγχάνει ειδικά ο Shor είναι η απουσία της αλγεβρικής (περιοδικής) δομής.

```
RSA: f(x) = 7^x mod 15 -> [1,7,4,13,1,7,4,13,...]  περίοδος 4  (ο QFT τη βρίσκει)
LWE: t = A.s + e  ->  μη περιοδική συνάρτηση· ο QFT δεν έχει περίοδο να βρει
```

### 8.3 Τι παραλείπει αυτό το παιχνίδι: ο μετασχηματισμός FO

Αυτό το notebook υλοποιεί τον **πυρήνα IND-CPA** (KeyGen / Κρυπτογράφηση / Αποκρυπτογράφηση).
Ο πραγματικός ML-KEM τον τυλίγει στον **μετασχηματισμό Fujisaki-Okamoto** για να φτάσει
IND-CCA2: η Ενθυλάκωση παράγει το κοινό μυστικό και τους σπόρους μαζί από `G(m || H(ek))`,
και η Αποενθυλάκωση **επανακρυπτογραφεί** και συγκρίνει, επιστρέφοντας ένα ψευδοτυχαίο
`J(z || c)` σε αναντιστοιχία (**σιωπηρή απόρριψη**). Αυτά, που παραλείπονται εδώ για σαφήνεια,
είναι που καθιστούν ένα αναπτυγμένο KEM ασφαλές έναντι επιθέσεων επιλεγμένου κρυπτοκειμένου.

### 8.4 Επίπεδα ασφαλείας (κατά προσέγγιση)

Χονδρικές εκτιμήσεις core-SVP (εξαρτώνται από το μοντέλο, όχι ακριβείς μετρήσεις πράξεων):
Kyber-512 ~ Κατηγορία 1, Kyber-768 ~ Κατηγορία 3, Kyber-1024 ~ Κατηγορία 5.

### Άσκηση 8: Εκτελέστε την πλήρη ροή και επαληθεύστε από άκρη σε άκρη


In [ ]:
print("╔══════════════════════════════════════════════════════════╗")
print("║   CRYSTALS-Kyber  Πλήρης Δοκιμή Συστήματος              ║")
print(f"║   q={Q}  n={N}  k={K}  ETA={ETA}                                    ║")
print("╚══════════════════════════════════════════════════════════╝\n")

# Δημιουργία Κλειδιού
print("── KeyGen ──────────────────────────────────────────────────")
s_ntt_kg = [ntt(S0, ROOTS, Q), ntt(S1, ROOTS, Q)]
e_ntt_kg = [ntt(E0, ROOTS, Q), ntt(E1, ROOTS, Q)]
t_hat_kg = keygen_t(A, s_ntt_kg, e_ntt_kg, Q)
assert t_hat_kg[0]==T0 and t_hat_kg[1]==T1
print(f"  t̂[0] = {t_hat_kg[0]}")
print(f"  t̂[1] = {t_hat_kg[1]}")
print("  ✓ Δημιουργία Κλειδιού ΟΚ\n")

# Ενθυλάκωση
print("── Encapsulation ───────────────────────────────────────────")
print(f"  μήνυμα m = {MSG}")
u_enc, v_enc = encapsulate(
    A, t_hat_kg, [R0,R1], [E1_0,E1_1], E2, MSG,
    ROOTS, INV_ROOTS, Q, K, N_INV
)
assert u_enc[0]==U0 and u_enc[1]==U1 and v_enc==V
print(f"  u[0] = {u_enc[0]}")
print(f"  u[1] = {u_enc[1]}")
print(f"  v    = {v_enc}")
print("  ✓ Ενθυλάκωση ΟΚ\n")

# Αποενθυλάκωση
print("── Decapsulation ───────────────────────────────────────────")
m_recovered = decapsulate(s_ntt_kg, u_enc, v_enc, ROOTS, INV_ROOTS, Q, K, N_INV)
assert m_recovered == MSG
print(f"  w       = {W}")
print(f"  αποκωδ. = {m_recovered}")
print(f"  αρχικό  = {MSG}")
print("  ✓ Αποενθυλάκωση ΟΚ\n")

# Αναφορά θορύβου
print("── Noise Budget ────────────────────────────────────────────")
for i in range(N):
    noise = (W[i] - M_HAT[i]) % Q
    if noise > Q//2: noise -= Q
    bar = "█" * abs(noise) + "░" * (Q//4 - abs(noise))
    print(f"  coeff[{i}]  w={W[i]:2d}  m̂={M_HAT[i]}  noise={noise:+2d}  [{bar}]  < q/4={Q//4}")

print("\n╔══════════════════════════════════════════════════════════╗")
print("║    ΠΛΗΡΗΣ ΔΟΚΙΜΗ ΣΥΣΤΗΜΑΤΟΣ ΕΠΙΤΥΧΗΣ                  ║")
print("║  Δημιουργία Κλειδιού → Ενθυλάκωση → Αποενθυλάκωση  ✓   ║")
print("╚══════════════════════════════════════════════════════════╝")